# Jour 3 — Mini-projet : localiser un objet dans une image

## Objectifs

- générer des images et des boîtes cohérentes
- entraîner un CNN de régression à quatre sorties
- calculer l’Intersection over Union
- inspecter visuellement les prédictions et leurs limites


## Mission

Nous générons de petites images contenant exactement un carré clair. Le modèle doit prédire les quatre coordonnées de sa boîte englobante.

![Pipeline du mini-projet de localisation](../ressources/illustrations/jour_03/04_pipeline_localisation.png)

Les données synthétiques offrent un environnement contrôlé : chaque boîte réelle est connue sans annotation manuelle. Elles permettent de comprendre le mécanisme, mais restent beaucoup plus simples que des images réelles.

Le projet suit les mêmes règles que les précédents : séparation avant l’apprentissage, validation pour piloter, test final et inspection des erreurs.


## La cible de régression

![Coordonnées d’une boîte englobante](../ressources/illustrations/jour_03/04_boite_englobante.png)

La cible `[x1, y1, x2, y2]` utilise des coordonnées normalisées entre 0 et 1. La dernière couche possède donc quatre neurones avec une activation `sigmoid`.

La loss MAE traite l’écart de chaque coordonnée. Une petite MAE ne garantit toutefois pas un bon recouvrement : des erreurs coordonnées différemment peuvent déplacer ou déformer la boîte.

Nous utiliserons donc aussi une métrique géométrique, l’IoU.


![Cibles et losses selon le type de sortie](../ressources/illustrations/jour_03/00_sorties_cibles.png)

### Contraindre une boîte valide

Une boîte cohérente vérifie `x1 < x2` et `y1 < y2`. Une couche Sigmoid garantit seulement que les quatre valeurs restent entre 0 et 1 ; elle ne garantit pas leur ordre.

Dans ce mini-projet simple, le modèle apprend généralement l’ordre grâce aux exemples. La fonction IoU remet néanmoins les coins prédits dans l’ordre avant le calcul.

Dans un système plus complexe, on peut prédire le centre, la largeur et la hauteur plutôt que deux coins. Chaque représentation possède ses avantages, mais la conversion et la convention doivent rester identiques dans les données, la loss, la métrique et l’affichage.


## Intersection over Union

![Intersection et union de deux boîtes](../ressources/illustrations/jour_03/04_iou.png)

L’**IoU** divise l’aire d’intersection par l’aire de l’union.

- `IoU = 1` signifie que les boîtes sont identiques ;
- `IoU = 0` signifie qu’elles ne se recouvrent pas ;
- une valeur intermédiaire mesure le recouvrement partiel.

Un seuil comme `IoU ≥ 0,5` peut servir à compter les localisations acceptables. Ce seuil reste une convention dépendante du problème.


### Pourquoi MAE et IoU se complètent

La MAE moyenne quatre écarts de coordonnées. Elle aide l’optimiseur car elle varie progressivement. L’IoU juge le recouvrement final et correspond mieux à la qualité géométrique visible.

Pour un petit objet, une erreur de quelques pixels peut faire fortement chuter l’IoU. Pour un grand objet, le même déplacement relatif a parfois moins d’effet. Il faut donc lire la métrique avec la taille des objets et inspecter les exemples.

Enfin, les carrés synthétiques ont un contraste, une forme et un arrière-plan contrôlés. Réussir ici démontre le mécanisme de localisation, pas une robustesse générale sur des images naturelles.


## Générer les données sans incohérence

Pour chaque image, nous tirons une taille et une position valides. Le carré reste entièrement dans l’image. Un léger bruit rend le problème moins trivial.

La fonction renvoie ensemble les images et les boîtes afin que leur ordre reste aligné. La graine rend la génération reproductible.


In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import tensorflow as tf

tf.keras.utils.set_random_seed(42)
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass

def make_square_dataset(n_samples, seed):
    rng = np.random.default_rng(seed)
    images = np.zeros((n_samples, 32, 32, 1), dtype="float32")
    boxes = np.zeros((n_samples, 4), dtype="float32")

    for i in range(n_samples):
        size = int(rng.integers(5, 13))
        x1 = int(rng.integers(1, 32 - size))
        y1 = int(rng.integers(1, 32 - size))
        x2, y2 = x1 + size, y1 + size
        intensity = float(rng.uniform(0.70, 1.0))

        images[i, y1:y2, x1:x2, 0] = intensity
        images[i, :, :, 0] += rng.normal(0, 0.025, size=(32, 32))
        images[i] = np.clip(images[i], 0, 1)
        boxes[i] = [x1 / 32, y1 / 32, x2 / 32, y2 / 32]

    return images, boxes

X_train, y_train = make_square_dataset(1000, seed=1)
X_validation, y_validation = make_square_dataset(250, seed=2)
X_test, y_test = make_square_dataset(250, seed=3)

fig, axes = plt.subplots(1, 5, figsize=(11, 2.3))
for ax, image, box_values in zip(axes, X_train[:5], y_train[:5]):
    x1, y1, x2, y2 = box_values * 32
    ax.imshow(image[:, :, 0], cmap="gray", vmin=0, vmax=1)
    ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor="lime", linewidth=2))
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Train :", X_train.shape, y_train.shape)
print("Validation :", X_validation.shape, y_validation.shape)
print("Test :", X_test.shape, y_test.shape)


## Un CNN à quatre sorties

Les convolutions extraient des caractéristiques spatiales. Après `Flatten`, une couche Dense combine ces caractéristiques et la dernière couche prédit les quatre coordonnées.

Ce modèle suppose qu’il existe exactement un objet par image. Une image vide ou plusieurs objets nécessiteraient une autre représentation de sortie.

L’arrêt anticipé surveille `val_loss`. Le test ne participe ni au choix de l’époque ni à la correction des poids.


In [ ]:
localizer = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 1)),
    tf.keras.layers.Conv2D(8, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(4, activation="sigmoid"),
])
localizer.compile(optimizer="adam", loss="mae", metrics=["mae"])

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=4, restore_best_weights=True
)
history = localizer.fit(
    X_train, y_train,
    validation_data=(X_validation, y_validation),
    epochs=25,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)
print("Époques effectuées :", len(history.history["loss"]))


## Calculer l’IoU sans masquer les cas difficiles

Les coordonnées prédites peuvent parfois arriver dans un ordre imparfait. La fonction remet d’abord les coins dans l’ordre avec `minimum` et `maximum`, puis calcule intersection et union.

Nous affichons la moyenne, la médiane et le 10e percentile. Le percentile bas attire l’attention sur les cas faibles que la moyenne pourrait masquer.


In [ ]:
def box_iou(true_boxes, predicted_boxes):
    tx1, ty1, tx2, ty2 = true_boxes.T
    px_a, py_a, px_b, py_b = predicted_boxes.T
    px1, px2 = np.minimum(px_a, px_b), np.maximum(px_a, px_b)
    py1, py2 = np.minimum(py_a, py_b), np.maximum(py_a, py_b)

    ix1, iy1 = np.maximum(tx1, px1), np.maximum(ty1, py1)
    ix2, iy2 = np.minimum(tx2, px2), np.minimum(ty2, py2)
    intersection = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)

    true_area = np.maximum(0, tx2 - tx1) * np.maximum(0, ty2 - ty1)
    pred_area = np.maximum(0, px2 - px1) * np.maximum(0, py2 - py1)
    union = true_area + pred_area - intersection
    return intersection / np.maximum(union, 1e-8)

predicted_boxes = localizer.predict(X_test, verbose=0)
ious = box_iou(y_test, predicted_boxes)

print("MAE test :", round(float(np.mean(np.abs(y_test - predicted_boxes))), 4))
print("IoU moyenne :", round(float(np.mean(ious)), 3))
print("IoU médiane :", round(float(np.median(ious)), 3))
print("10e percentile :", round(float(np.percentile(ious, 10)), 3))

plt.figure(figsize=(7, 3.5))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.xlabel("Époque")
plt.ylabel("MAE")
plt.title("Apprentissage des coordonnées")
plt.legend()
plt.show()


## Superposer vérité et prédiction

Le vert représente la boîte réelle et le rouge la boîte prédite. L’IoU dans le titre relie la mesure numérique à la situation visible.

Une erreur peut venir d’un bord flou, du bruit, d’une petite taille d’objet ou d’une prédiction légèrement décalée. L’inspection évite d’interpréter toutes les faibles IoU comme un seul type de problème.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, index in zip(axes.ravel(), range(8)):
    true_box = y_test[index] * 32
    pred_box = predicted_boxes[index] * 32
    tx1, ty1, tx2, ty2 = true_box
    px1, py1, px2, py2 = pred_box

    ax.imshow(X_test[index, :, :, 0], cmap="gray", vmin=0, vmax=1)
    ax.add_patch(Rectangle((tx1, ty1), tx2-tx1, ty2-ty1, fill=False, edgecolor="lime", linewidth=2))
    ax.add_patch(Rectangle((px1, py1), px2-px1, py2-py1, fill=False, edgecolor="red", linewidth=2))
    ax.set_title(f"IoU {ious[index]:.2f}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## À vous de jouer — analyser les localisations

Calculez la proportion de tests dont l’IoU est au moins égale à 0,5.

Trouvez ensuite l’index de la plus faible IoU et affichez cette image avec la boîte réelle en vert et la prédiction en rouge. Rédigez une phrase décrivant l’erreur visible.


In [ ]:
# Calculez le taux IoU >= 0.5 puis affichez le pire exemple.
pass


## Bilan du second jour Deep Learning

Nous sommes passés d’une sortie de classification à une sortie géométrique. Le réseau ne choisit plus seulement une catégorie : il apprend quatre coordonnées continues.

La démarche reste la même : définir précisément la cible, séparer les données, choisir une loss compatible, contrôler la validation, mesurer avec une métrique adaptée et inspecter les erreurs.

Ce mini-projet reste volontairement contrôlé. Des images contenant plusieurs objets, des tailles très variées ou des arrière-plans complexes demandent une formulation plus riche et davantage de données.
